# 02 — EDA : Identification des conditions d'entrée optimales
**Savvy Crypto Agent — Jedha Bootcamp Final Project**

Objectif : trouver les combinaisons de conditions qui font passer le win rate de 9% (aléatoire) à >67% (seuil de rentabilité avec TP=0.7%, SL=0.5%, frais=0.3%).

> **Règle absolue : tout ce notebook travaille exclusivement sur `train_features.csv`. Le test set n'est pas touché.**

## 0. Imports & chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import json, os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('husl')

PROCESSED = "C:\\Users\\casir\\Desktop\\CRYPTOS\\data\\processed\\"

# --- latest.json lookup (fallback sans suffixe au premier lancement) ---
if os.path.exists(PROCESSED + 'latest.json'):
    with open(PROCESSED + 'latest.json') as f:
        latest = json.load(f)
    print(f"Chargement version : {latest['date']}")
else:
    latest = {'train': 'train_features.csv', 'test': 'test_features.csv',
              'feature_cols': 'feature_cols.json', 'config': 'config.json', 'date': 'fallback'}
    print('latest.json absent -> fallback sans suffixe')

with open(PROCESSED + latest['config']) as f:
    cfg = json.load(f)

TP_NET            = cfg['tp_net']
SL_NET            = cfg['sl_net']
BREAKEVEN_WINRATE = cfg['breakeven']

print(f'Config chargee depuis {latest["config"]} :')
print(f'  TP net : {TP_NET*100:.2f}%')
print(f'  SL net : {SL_NET*100:.2f}%')
print(f'  Seuil rentabilite : {BREAKEVEN_WINRATE*100:.1f}%')

train = pd.read_csv(PROCESSED + latest['train'])
train['candle_time'] = pd.to_datetime(train['candle_time'], utc=True)

FULL_SYMBOLS = ['ADA-EUR','AVAX-EUR','BCH-EUR','DOT-EUR','LINK-EUR','LTC-EUR','SOL-EUR']
train_full = train[train['symbol'].isin(FULL_SYMBOLS)].copy()

print(f'\nTRAIN charge ({latest["train"]}) : {len(train):,} bougies')
print(f'Win rate baseline : {train["label"].mean()*100:.1f}%')

## 1. Win rate par zone de RSI
La corrélation RSI → label est de -0.35. Quantifions l'effet.

In [ ]:
bins_rsi = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
labels_rsi = ['0-10','10-20','20-30','30-40','40-50','50-60','60-70','70-80','80-90','90-100']

train['rsi_zone'] = pd.cut(train['rsi14'], bins=bins_rsi, labels=labels_rsi)
rsi_stats = train.groupby('rsi_zone', observed=True).agg(
    n        = ('label', 'count'),
    win_rate = ('label', 'mean'),
    pct_data = ('label', lambda x: len(x)/len(train)*100)
).reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

colors = ['steelblue' if w > BREAKEVEN_WINRATE else 'tomato' for w in rsi_stats['win_rate']]
bars = ax1.bar(rsi_stats['rsi_zone'], rsi_stats['win_rate']*100, color=colors, edgecolor='white')
ax1.axhline(BREAKEVEN_WINRATE*100, color='black', linestyle='--', linewidth=1.5, label=f'Seuil rentabilité ({BREAKEVEN_WINRATE*100:.0f}%)')
ax1.axhline(train['label'].mean()*100, color='gray', linestyle=':', linewidth=1.2, label=f'Baseline ({train["label"].mean()*100:.1f}%)')
ax1.set_title('Win rate par zone de RSI14', fontsize=13)
ax1.set_xlabel('Zone RSI')
ax1.set_ylabel('Win rate (%)')
ax1.legend()
for bar, (_, row) in zip(bars, rsi_stats.iterrows()):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{row["win_rate"]*100:.1f}%', ha='center', fontsize=8)

ax2.bar(rsi_stats['rsi_zone'], rsi_stats['pct_data'], color='steelblue', alpha=0.7)
ax2.set_title('% du dataset dans chaque zone RSI', fontsize=13)
ax2.set_xlabel('Zone RSI')
ax2.set_ylabel('% des bougies')

plt.tight_layout()
plt.show()

print(rsi_stats.to_string(index=False))

## 2. Win rate par zone de Stochastique
Corrélation stoch_k → label = -0.43 (plus forte que RSI)

In [ ]:
bins_stoch = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
train['stoch_zone'] = pd.cut(train['stoch_k'], bins=bins_stoch, labels=labels_rsi)

stoch_stats = train.groupby('stoch_zone', observed=True).agg(
    n=('label','count'), win_rate=('label','mean')
).reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['steelblue' if w > BREAKEVEN_WINRATE else 'tomato' for w in stoch_stats['win_rate']]
bars = ax.bar(stoch_stats['stoch_zone'], stoch_stats['win_rate']*100, color=colors, edgecolor='white')
ax.axhline(BREAKEVEN_WINRATE*100, color='black', linestyle='--', linewidth=1.5, label=f'Seuil rentabilité')
ax.axhline(train['label'].mean()*100, color='gray', linestyle=':', linewidth=1.2, label='Baseline')
ax.set_title('Win rate par zone de Stochastique K', fontsize=13)
ax.set_xlabel('Zone Stoch K')
ax.set_ylabel('Win rate (%)')
ax.legend()
for bar, (_, row) in zip(bars, stoch_stats.iterrows()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{row["win_rate"]*100:.1f}%', ha='center', fontsize=8)
plt.tight_layout()
plt.show()
print(stoch_stats.to_string(index=False))

## 3. Win rate par zone de Bollinger Position
`bb_pos` : 0 = prix au bas de la bande, 1 = prix en haut. Corrélation -0.29.

In [ ]:
bins_bb = [-0.5, 0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.5]
labels_bb = ['<0','0-0.1','0.1-0.2','0.2-0.3','0.3-0.4','0.4-0.5','0.5-0.6','0.6-0.7','0.7-0.8','0.8-0.9','0.9-1.0','>1.0']

train['bb_zone'] = pd.cut(train['bb_pos'].clip(-0.5, 1.5), bins=bins_bb, labels=labels_bb)
bb_stats = train.groupby('bb_zone', observed=True).agg(
    n=('label','count'), win_rate=('label','mean')
).reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['steelblue' if w > BREAKEVEN_WINRATE else 'tomato' for w in bb_stats['win_rate']]
bars = ax.bar(bb_stats['bb_zone'], bb_stats['win_rate']*100, color=colors, edgecolor='white')
ax.axhline(BREAKEVEN_WINRATE*100, color='black', linestyle='--', linewidth=1.5)
ax.axhline(train['label'].mean()*100, color='gray', linestyle=':')
ax.set_title('Win rate par position dans les Bollinger Bands', fontsize=13)
ax.set_xlabel('Position bb_pos (0=bas, 1=haut)')
ax.set_ylabel('Win rate (%)')
for bar, (_, row) in zip(bars, bb_stats.iterrows()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{row["win_rate"]*100:.1f}%', ha='center', fontsize=7)
plt.tight_layout()
plt.show()

## 4. Patterns horaires — à quelle heure UTC gagne-t-on le plus ?

In [ ]:
hourly = train.groupby('hour_utc').agg(
    n=('label','count'), win_rate=('label','mean')
).reset_index()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

colors = ['steelblue' if w > BREAKEVEN_WINRATE else 'tomato' for w in hourly['win_rate']]
ax1.bar(hourly['hour_utc'], hourly['win_rate']*100, color=colors, edgecolor='white')
ax1.axhline(BREAKEVEN_WINRATE*100, color='black', linestyle='--', linewidth=1.5, label='Seuil rentabilité')
ax1.axhline(train['label'].mean()*100, color='gray', linestyle=':', label='Baseline')
ax1.set_title('Win rate par heure UTC', fontsize=13)
ax1.set_ylabel('Win rate (%)')
ax1.legend()

# Zones de session
ax1.axvspan(0, 8, alpha=0.05, color='blue', label='Asie')
ax1.axvspan(8, 16, alpha=0.05, color='green', label='Europe')
ax1.axvspan(13, 22, alpha=0.05, color='orange', label='US')

ax2.bar(hourly['hour_utc'], hourly['n'], color='steelblue', alpha=0.6)
ax2.set_title('Volume de bougies par heure UTC', fontsize=13)
ax2.set_xlabel('Heure UTC')
ax2.set_ylabel('Nombre de bougies')
ax2.set_xticks(range(24))

plt.tight_layout()
plt.show()

best_hours = hourly[hourly['win_rate'] > BREAKEVEN_WINRATE].sort_values('win_rate', ascending=False)
print('Heures avec win rate > seuil de rentabilité :')
print(best_hours[['hour_utc','win_rate','n']].to_string(index=False))

## 5. Win rate par jour de semaine

In [ ]:
dow_names = {0:'Lundi', 1:'Mardi', 2:'Mercredi', 3:'Jeudi', 4:'Vendredi', 5:'Samedi', 6:'Dimanche'}
dow_stats = train.groupby('dow').agg(
    n=('label','count'), win_rate=('label','mean')
).reset_index()
dow_stats['day_name'] = dow_stats['dow'].map(dow_names)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue' if w > BREAKEVEN_WINRATE else 'tomato' for w in dow_stats['win_rate']]
bars = ax.bar(dow_stats['day_name'], dow_stats['win_rate']*100, color=colors, edgecolor='white')
ax.axhline(BREAKEVEN_WINRATE*100, color='black', linestyle='--', linewidth=1.5)
ax.axhline(train['label'].mean()*100, color='gray', linestyle=':')
ax.set_title('Win rate par jour de semaine', fontsize=13)
ax.set_ylabel('Win rate (%)')
for bar, (_, row) in zip(bars, dow_stats.iterrows()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, f'{row["win_rate"]*100:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 6. Win rate par symbole

In [ ]:
# Deriver exit_ret depuis outcome (approximation conservative)
# wins trend (+2%) et wins MR (+0.85%) sont tous mappes a TP_NET
# -> PnL sous-estime pour les wins trend, mais direction correcte

# Charger test (pas charge dans la cellule d'imports)
test = pd.read_csv(PROCESSED + latest['test'])
test['candle_time'] = pd.to_datetime(test['candle_time'], utc=True)

train['exit_ret'] = train['outcome'].map({
    'win':     TP_NET,
    'loss':    -SL_NET,
    'timeout': 0.0
})
test['exit_ret'] = test['outcome'].map({
    'win':     TP_NET,
    'loss':    -SL_NET,
    'timeout': 0.0
})
print(f'exit_ret derive - mean: {train["exit_ret"].mean()*100:.4f}% | '
      f'wins: {(train["exit_ret"]==TP_NET).sum():,} | '
      f'losses: {(train["exit_ret"]==-SL_NET).sum():,}')


In [ ]:
sym_stats = train.groupby('symbol').agg(
    n=('label','count'),
    win_rate=('label','mean'),
    expected_ret=('exit_ret','mean')
).sort_values('win_rate', ascending=False).reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

colors = ['steelblue' if w > BREAKEVEN_WINRATE else 'tomato' for w in sym_stats['win_rate']]
bars = ax1.barh(sym_stats['symbol'], sym_stats['win_rate']*100, color=colors)
ax1.axvline(BREAKEVEN_WINRATE*100, color='black', linestyle='--', linewidth=1.5)
ax1.axvline(train['label'].mean()*100, color='gray', linestyle=':')
ax1.set_title('Win rate par symbole (TRAIN)', fontsize=13)
ax1.set_xlabel('Win rate (%)')
for bar, (_, row) in zip(bars, sym_stats.iterrows()):
    ax1.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2, f'{row["win_rate"]*100:.1f}%', va='center', fontsize=9)

ax2.barh(sym_stats['symbol'], sym_stats['expected_ret']*100, color=['steelblue' if r>0 else 'tomato' for r in sym_stats['expected_ret']])
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_title('Rendement moyen par trade par symbole (%)', fontsize=13)
ax2.set_xlabel('Rendement moyen (%)')

plt.tight_layout()
plt.show()
print(sym_stats.to_string(index=False))

## 7. Analyse des combinaisons de conditions — le coeur de l'EDA

On teste toutes les combinaisons de seuils pour trouver les règles qui maximisent le win rate tout en gardant un nombre de trades suffisant.

In [ ]:
def filter_stats(mask, label='filtre', baseline_n=len(train)):
    """Calcule les stats d'un filtre donné."""
    sub = train[mask]
    if len(sub) == 0:
        return None
    wr  = sub['label'].mean()
    n   = len(sub)
    pct = n / baseline_n * 100
    expected = wr * TP_NET + (1-wr) * (-SL_NET)
    return {
        'filtre'    : label,
        'n'         : n,
        'pct_data'  : round(pct, 1),
        'win_rate'  : round(wr*100, 2),
        'ev_per_trade': round(expected*100, 3),  # expected value en %
        'rentable'  : wr > BREAKEVEN_WINRATE
    }

results = []

# --- RSI seul ---
for thresh in [20, 25, 30, 35, 40]:
    results.append(filter_stats(train['rsi14'] < thresh, f'RSI < {thresh}'))

# --- Stochastique seul ---
for thresh in [10, 15, 20, 25, 30]:
    results.append(filter_stats(train['stoch_k'] < thresh, f'Stoch_K < {thresh}'))

# --- RSI + Stoch ---
for r in [30, 35, 40]:
    for s in [20, 30]:
        results.append(filter_stats(
            (train['rsi14'] < r) & (train['stoch_k'] < s),
            f'RSI<{r} & Stoch<{s}'
        ))

# --- RSI + EMA50 ---
for r in [30, 35, 40]:
    for e in [-0.5, -1.0, -2.0]:
        results.append(filter_stats(
            (train['rsi14'] < r) & (train['price_vs_ema50'] < e),
            f'RSI<{r} & PvsEMA50<{e}%'
        ))

# --- RSI + Stoch + EMA ---
for r in [30, 35, 40]:
    for s in [20, 30]:
        results.append(filter_stats(
            (train['rsi14'] < r) & (train['stoch_k'] < s) & (train['price_vs_ema50'] < -0.5),
            f'RSI<{r} & Stoch<{s} & PvsEMA50<-0.5%'
        ))

# --- BB position ---
for b in [0.2, 0.3, 0.4]:
    results.append(filter_stats(train['bb_pos'] < b, f'bb_pos < {b}'))

# --- RSI + BB ---
for r in [30, 40]:
    for b in [0.2, 0.3]:
        results.append(filter_stats(
            (train['rsi14'] < r) & (train['bb_pos'] < b),
            f'RSI<{r} & bb_pos<{b}'
        ))

# --- EMA alignment bearish ---
results.append(filter_stats(train['ema_aligned_bear'] == 1, 'EMA bearish aligned'))
results.append(filter_stats(
    (train['rsi14'] < 35) & (train['ema_aligned_bear'] == 1),
    'RSI<35 & EMA bearish'
))

df_results = pd.DataFrame([r for r in results if r is not None])
df_results = df_results.sort_values('win_rate', ascending=False)

print(f'Baseline win rate : {train["label"].mean()*100:.1f}%')
print(f'Seuil rentabilité : {BREAKEVEN_WINRATE*100:.1f}%')
print()
print(df_results.to_string(index=False))

In [ ]:
# Visualisation des combinaisons
top = df_results.head(25).sort_values('win_rate')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 10))

colors = ['steelblue' if r else 'tomato' for r in top['rentable']]
bars = ax1.barh(top['filtre'], top['win_rate'], color=colors, edgecolor='white')
ax1.axvline(BREAKEVEN_WINRATE*100, color='black', linestyle='--', linewidth=1.5, label='Seuil rentabilité')
ax1.axvline(train['label'].mean()*100, color='gray', linestyle=':', label='Baseline')
ax1.set_title('Win rate des combinaisons de conditions', fontsize=13)
ax1.set_xlabel('Win rate (%)')
ax1.legend()
for bar, (_, row) in zip(bars, top.iterrows()):
    ax1.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
             f'{row["win_rate"]}% (n={row["n"]:,})', va='center', fontsize=7)

colors2 = ['steelblue' if r else 'tomato' for r in top['rentable']]
ax2.barh(top['filtre'], top['ev_per_trade'], color=colors2, edgecolor='white')
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_title('Expected value par trade (%)', fontsize=13)
ax2.set_xlabel('EV par trade (%)')

plt.tight_layout()
plt.show()

## 8. Analyse de la meilleure règle identifiée
On prend la meilleure combinaison et on l'analyse en détail.

In [ ]:
# Affiche la meilleure règle avec n suffisant (>500 trades)
best = df_results[df_results['n'] > 500].iloc[0]
print(f'Meilleure règle (n > 500) :')
print(f'  Filtre     : {best["filtre"]}')
print(f'  Win rate   : {best["win_rate"]}%')
print(f'  N trades   : {best["n"]:,}')
print(f'  EV / trade : {best["ev_per_trade"]}%')
print(f'  Rentable   : {best["rentable"]}')

In [ ]:
# Evolution temporelle du win rate de la meilleure règle
# On vérifie que le signal est stable dans le temps (pas juste sur une période)

# Utilise RSI < 35 & Stoch < 20 comme exemple — adapte si la meilleure règle est différente
BEST_MASK = (train['rsi14'] < 35) & (train['stoch_k'] < 20)

train['month'] = train['candle_time'].dt.to_period('M')
monthly = train[BEST_MASK].groupby('month').agg(
    n=('label','count'), win_rate=('label','mean')
).reset_index()
monthly['month_str'] = monthly['month'].astype(str)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(monthly['month_str'], monthly['win_rate']*100, marker='o', color='steelblue', linewidth=2)
ax1.axhline(BREAKEVEN_WINRATE*100, color='black', linestyle='--', linewidth=1.5, label='Seuil rentabilité')
ax1.axhline(train['label'].mean()*100, color='gray', linestyle=':', label='Baseline')
ax1.fill_between(monthly['month_str'], BREAKEVEN_WINRATE*100, monthly['win_rate']*100,
                  where=monthly['win_rate']>BREAKEVEN_WINRATE, alpha=0.2, color='steelblue', label='Rentable')
ax1.fill_between(monthly['month_str'], BREAKEVEN_WINRATE*100, monthly['win_rate']*100,
                  where=monthly['win_rate']<=BREAKEVEN_WINRATE, alpha=0.2, color='tomato', label='Pas rentable')
ax1.set_title('Evolution mensuelle du win rate — RSI<35 & Stoch<20', fontsize=13)
ax1.set_ylabel('Win rate (%)')
ax1.legend()
plt.xticks(rotation=45)

ax2.bar(monthly['month_str'], monthly['n'], color='steelblue', alpha=0.7)
ax2.set_title('Nombre de trades par mois (filtre actif)', fontsize=13)
ax2.set_ylabel('Nombre de trades')
ax2.set_xlabel('Mois')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 9. Heatmap RSI × Stochastique — carte complète des win rates

In [ ]:
train['rsi_bin']   = pd.cut(train['rsi14'],   bins=[0,20,30,40,50,60,70,100], labels=['<20','20-30','30-40','40-50','50-60','60-70','>70'])
train['stoch_bin'] = pd.cut(train['stoch_k'], bins=[0,10,20,30,40,50,60,70,80,90,100],
                            labels=['<10','10-20','20-30','30-40','40-50','50-60','60-70','70-80','80-90','>90'])

heatmap_data = train.groupby(['rsi_bin', 'stoch_bin'], observed=True)['label'].mean().unstack() * 100
heatmap_n    = train.groupby(['rsi_bin', 'stoch_bin'], observed=True)['label'].count().unstack()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn',
            vmin=0, vmax=30, ax=ax1, cbar_kws={'label': 'Win rate (%)'})
ax1.set_title('Win rate (%) — RSI × Stochastique', fontsize=13)
ax1.set_xlabel('Stoch K zone')
ax1.set_ylabel('RSI zone')

sns.heatmap(heatmap_n, annot=True, fmt=',', cmap='Blues', ax=ax2, cbar_kws={'label': 'N bougies'})
ax2.set_title('Nombre de bougies — RSI × Stochastique', fontsize=13)
ax2.set_xlabel('Stoch K zone')
ax2.set_ylabel('RSI zone')

plt.tight_layout()
plt.show()

## 10. Backtest simple de la meilleure règle (sur TRAIN)
Simulation de la courbe de capital avec la règle identifiée.

In [ ]:
CAPITAL_INITIAL = 10_000
TRADE_SIZE      = 500

# Applique le filtre sur le train — 1 trade par bougie filtrée
trades = train[BEST_MASK].copy().sort_values('candle_time')

# Calcul du capital
trades['pnl_eur']  = trades['exit_ret'] * TRADE_SIZE
trades['capital']  = CAPITAL_INITIAL + trades['pnl_eur'].cumsum()
trades['drawdown'] = trades['capital'] / trades['capital'].cummax() - 1

# Stats globales
total_trades = len(trades)
win_trades   = (trades['label'] == 1).sum()
total_pnl    = trades['pnl_eur'].sum()
final_cap    = CAPITAL_INITIAL + total_pnl
max_dd       = trades['drawdown'].min() * 100
pct_return   = total_pnl / CAPITAL_INITIAL * 100

print(f'=== BACKTEST TRAIN — {best["filtre"]} ===')
print(f'Capital initial   : {CAPITAL_INITIAL:,} €')
print(f'Taille trade      : {TRADE_SIZE} €')
print(f'Total trades      : {total_trades:,}')
print(f'Win trades        : {win_trades:,} ({win_trades/total_trades*100:.1f}%)')
print(f'PnL total         : {total_pnl:+,.0f} €')
print(f'Capital final     : {final_cap:,.0f} €')
print(f'Rendement total   : {pct_return:+.1f}%')
print(f'Max drawdown      : {max_dd:.1f}%')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(trades['candle_time'], trades['capital'], color='steelblue', linewidth=1.5)
ax1.axhline(CAPITAL_INITIAL, color='gray', linestyle=':', linewidth=1)
ax1.set_title(f'Courbe de capital — {best["filtre"]} (TRAIN)', fontsize=13)
ax1.set_ylabel('Capital (€)')
ax1.fill_between(trades['candle_time'], CAPITAL_INITIAL, trades['capital'],
                  where=trades['capital']>=CAPITAL_INITIAL, alpha=0.15, color='steelblue')
ax1.fill_between(trades['candle_time'], CAPITAL_INITIAL, trades['capital'],
                  where=trades['capital']<CAPITAL_INITIAL, alpha=0.15, color='tomato')

ax2.fill_between(trades['candle_time'], trades['drawdown']*100, 0, color='tomato', alpha=0.4)
ax2.set_title('Drawdown (%)', fontsize=13)
ax2.set_ylabel('Drawdown (%)')
ax2.set_xlabel('Date')

plt.tight_layout()
plt.show()

## 11. Synthèse EDA — ce qu'on retient

In [ ]:
print('=' * 60)
print('SYNTHÈSE EDA — RÈGLES À RETENIR POUR LE ML')
print('=' * 60)
print()
print('STRATÉGIE IDENTIFIÉE : Mean Reversion (retour à la moyenne)')
print('Les trades gagnants surviennent après une survente marquée.')
print()
print('FEATURES LES PLUS DISCRIMINANTES :')
print('  1. stoch_k / stoch_d  (corr -0.43 / -0.42)')
print('  2. rsi14              (corr -0.35)')
print('  3. price_vs_ema50     (corr -0.30)')
print('  4. bb_pos             (corr -0.29)')
print('  5. ema_aligned_bear   (corr +0.34) → tendance baissière = rebond potentiel')
print()
print('CONDITIONS D ENTRÉE CANDIDATES (à valider par le ML) :')
top5 = df_results[df_results['n'] > 200].head(5)
for _, row in top5.iterrows():
    print(f'  • {row["filtre"]:40s} win={row["win_rate"]}%  n={row["n"]:,}')
print()
print('FILTRES TEMPORELS POTENTIELS :')
best_h = hourly[hourly['win_rate'] > BREAKEVEN_WINRATE]['hour_utc'].tolist()
print(f'  Heures favorables UTC : {best_h}')
print()
print('PROCHAINE ÉTAPE : notebook 03_ml_model.ipynb')
print('  → XGBoost / Random Forest sur les 47 features')
print('  → Feature importance pour confirmer l EDA')
print('  → Calibration du seuil de confiance')
print('=' * 60)

In [ ]:
# Sauvegarde des resultats pour le notebook ML
import json
from datetime import datetime

DATE = datetime.now().strftime('%Y%m%d')

# Versionne
df_results.to_csv(PROCESSED + f'eda_filter_results_{DATE}.csv', index=False)
# Sans suffixe (compatibilite)
df_results.to_csv(PROCESSED + 'eda_filter_results.csv', index=False)

eda_summary = {
    'breakeven_winrate': round(BREAKEVEN_WINRATE, 4),
    'baseline_winrate' : round(train['label'].mean(), 4),
    'best_rule'        : best['filtre'],
    'best_winrate'     : best['win_rate'],
    'best_n'           : int(best['n']),
    'best_hours_utc'   : best_h,
    'top_features'     : ['stoch_k','stoch_d','rsi14','price_vs_ema50','bb_pos','ema_aligned_bear']
}

with open(PROCESSED + f'eda_summary_{DATE}.json', 'w') as f:
    json.dump(eda_summary, f, indent=2)
with open(PROCESSED + 'eda_summary.json', 'w') as f:
    json.dump(eda_summary, f, indent=2)

print('EDA sauvegardee :')
print(f'  eda_filter_results_{DATE}.csv (+ copie sans suffixe)')
print(f'  eda_summary_{DATE}.json (+ copie sans suffixe)')
print()
print('Notebook 02 termine. Passe au notebook 03_ml_model.ipynb')